# TruthLens text detector on a Colab GPU

Fine-tunes RoBERTa-base and DeBERTa-v3-small on MAGE, adds the better one to the statistical ensemble, and runs the paraphrase robustness test.

## Setup

The notebook clones the repository from GitHub, so push your local commits first. Only cells marked GPU only need a GPU runtime. Everything that must survive a disconnect is written to Google Drive.

In [ ]:
REPO = 'https://github.com/Shashank17singh/TruthLens.git'
BRANCH = 'main'
import os
if not os.path.exists('/content/TruthLens'):
    !git clone -q --branch {BRANCH} {REPO} /content/TruthLens
%cd /content/TruthLens
!git pull -q
assert os.path.exists('ml/text/train.py'), 'This branch does not have ml/text/train.py yet. Push your commits, then run this cell again.'

In [ ]:
!pip install -q -r ml/requirements.txt
import torch
GPU = torch.cuda.is_available()
print(torch.cuda.get_device_name(0) if GPU else 'No GPU: cells that need one will stop. Runtime, Change runtime type, T4 GPU.')

If pip asks you to restart the session, restart it and run the two cells above again. The clone is kept.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/truthlens'
os.makedirs(DRIVE, exist_ok=True)

## Data

Prepared text is cached on Drive, because language detection over MAGE takes a while.

In [ ]:
CACHE = f'{DRIVE}/text-data.tar'
if os.path.exists(CACHE):
    !tar -xf {CACHE}
else:
    for name in ['mage', 'hc3', 'detector_bias']:
        !python -m ml.common.prepare {name} --workers 4
    !tar -cf {CACHE} data/processed/mage data/processed/hc3 data/processed/detector_bias data/manifests
!ls data/manifests

## Fine-tune the transformers (GPU only)

Each command resumes from its last epoch if the session ends. `steps_per_epoch` keeps one epoch inside a session.

In [ ]:
CKPT = f'{DRIVE}/checkpoints/text'
if not GPU:
    print('Skipped: this cell needs a GPU runtime.')
else:
    !python -m ml.text.train model=roberta checkpoint_dir={CKPT} steps_per_epoch=3000 tracker=none

In [ ]:
if not GPU:
    print('Skipped: this cell needs a GPU runtime.')
else:
    !python -m ml.text.train model=deberta checkpoint_dir={CKPT} steps_per_epoch=3000 tracker=none

In [ ]:
if not GPU:
    print('Skipped: this cell needs a GPU runtime.')
else:
    import glob
    runs = {path: torch.load(path, map_location='cpu', weights_only=False)['best_auc'] for path in glob.glob(f'{CKPT}/*/*-best.pt')}
    for path, auc in runs.items():
        print(f'{auc:.4f}  {path}')
    BEST = max(runs, key=runs.get)
    print('using', BEST)

## Ensemble with the transformer logit (GPU only)

The same features as the CPU run, on larger samples, with the best transformer as one more column.

In [ ]:
if not GPU:
    print('Skipped: this cell needs a GPU runtime.')
else:
    FEATURES = f'{DRIVE}/features/text-ensemble'
    parts = [('mage-train', 'train', 5000), ('mage-val', 'val', 2000), ('mage-test', 'test', 4000), ('mage-unseen', 'unseen_test', 2000)]
    for name, split, limit in parts:
        !python -m ml.text.ensemble extract --manifest data/manifests/mage.csv --split {split} --limit {limit} --checkpoint {BEST} --output {FEATURES}/{name}.parquet
    !python -m ml.text.ensemble extract --manifest data/manifests/hc3.csv --limit 2000 --exclude-groups-of data/manifests/mage.csv --checkpoint {BEST} --output {FEATURES}/hc3.parquet
    !python -m ml.text.ensemble extract --manifest data/manifests/detector_bias.csv --checkpoint {BEST} --output {FEATURES}/detector_bias.parquet

In [ ]:
RESULTS = f'{DRIVE}/results/text-ensemble'
!python -m ml.text.ensemble fit --train {FEATURES}/mage-train.parquet --output {CKPT}/meta-ensemble.joblib
!python -m ml.text.ensemble evaluate --model {CKPT}/meta-ensemble.joblib --validation {FEATURES}/mage-val.parquet --test {FEATURES}/mage-test.parquet {FEATURES}/mage-unseen.parquet {FEATURES}/hc3.parquet {FEATURES}/detector_bias.parquet --output {RESULTS}

## Paraphrase robustness (CPU or GPU)

This section does not need the transformer cells, so it also runs on a CPU runtime. It uses the statistics-only ensemble, the one the service runs, with the T5 paraphraser. The paraphraser does not fit in 8 GB of memory next to GPT-2, which is why it runs here. Statistics-only features are extracted without `--checkpoint`.

In [ ]:
STATS = f'{DRIVE}/features/text-stats'
CKPT = f'{DRIVE}/checkpoints/text'
for name, split, limit in [('mage-train', 'train', 5000), ('mage-val', 'val', 2000), ('mage-test', 'test', 4000)]:
    !python -m ml.text.ensemble extract --manifest data/manifests/mage.csv --split {split} --limit {limit} --output {STATS}/{name}.parquet
!python -m ml.text.ensemble fit --train {STATS}/mage-train.parquet --output {CKPT}/meta.joblib
!python -m ml.text.ensemble evaluate --model {CKPT}/meta.joblib --validation {STATS}/mage-val.parquet --test {STATS}/mage-test.parquet --output {DRIVE}/results/text-stats
!python -m ml.text.robustness --model {CKPT}/meta.joblib --features {STATS}/mage-test.parquet --threshold-from {DRIVE}/results/text-stats/metrics.json --limit 300 --output {DRIVE}/results/text-stats/robustness.json

Copy `metrics.json` and `robustness.json` from Drive into `ml/eval/results/` in your local clone.